# E3 per-model seed honouring (CALIBRATION.md §3.6)

`E3_seed_verification.ipynb` closed P3-T5 and P0-T2 §3.5. It could not close
§3.6, and said so: `run_cell.R` positions a fixed L'Ecuyer substream in the
**global** RNG state immediately before every fit, so re-running a seed
reproduces a model's columns whether that model honoured its explicit seed
argument or ignored it and drew from the global stream. Both hypotheses
predict identical output, so the comparison has no power to separate them.

This notebook runs the comparison that does: **same model, same data, same
explicit seed, two different global RNG states.**

**No E3 replication is re-run.** Not one of the 4,000 stored rows is read.
The notebook generates one DGP1 `n = 100` dataset and issues nine fits, and
nothing about the shipped data depends on the answer: exact reproduction is
already an observed fact across ten hosts, two worker counts and two shard
offsets. What the answer decides is what the Data Availability statement may
attribute that reproducibility *to*.

Two of the four models named in §3.6 are out of scope by construction, and
source inspection settles that without spending a session. `fast_bart()`
takes no seed argument: its signature at `MVBCF_Code.cpp:659` has 18
parameters, none of them a seed, and every draw in the file goes through R's
own generator (`R::runif`, plus `rmvnorm`, `riwish` and `sample` from
RcppDist and Rcpp sugar), with the `Rcpp::export` `RNGScope` reading and
writing `.Random.seed`. There is no independent generator anywhere in it: no
`mt19937`, no `rand()`, no `arma::randu`/`randn`. `MultiskewBART` is likewise
called without a seed. Both are governed entirely by §3.3 stream positioning,
which the §3.5 four-way identity test already validated.

That leaves `stochtree::bcf` and `dbarts` (used directly for the propensity
fit and underneath `bartCause::bartc`).

**The verdict is asymmetric, so read it carefully.** A PASS is conclusive:
the explicit seed alone determines the fit. A FAIL does *not* mean the seed
argument is ignored, it means the fit also consumes the global stream, which
is ordinary behaviour for an R model wrapper and exactly the case §3.3
positioning exists to cover. Each block also carries a positive control at a
*different* explicit seed: if that comes back identical too, the fit is not
seed-sensitive at all and the result is reported as INCONCLUSIVE rather than
as a pass.

Runtime is about 20 minutes, most of it environment setup.

In [ ]:
import os
import sys
import subprocess

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA.git"
CANDIDATES = [
    os.path.abspath(os.path.join(os.getcwd(), "..")),          # notebooks/ in a checkout
    os.getcwd(),
    "/content/Test-Informed-Simulation-Count-Algorithm-TISCA",
    "/content/TISCA_repo",
]
REPO_ROOT = next((p for p in CANDIDATES
                  if os.path.isdir(os.path.join(p, "tisca", "python"))), None)
if REPO_ROOT is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TISCA_repo"], check=True)
    REPO_ROOT = "/content/TISCA_repo"
sys.path.insert(0, os.path.join(REPO_ROOT, "tisca", "python"))

RESULTS = os.path.join(REPO_ROOT, "results")
FIGURES = os.path.join(REPO_ROOT, "figures")
os.makedirs(FIGURES, exist_ok=True)
print("repo:", REPO_ROOT)


def download(path):
    """Colab download fallback (standing rule); a no-op off Colab."""
    try:
        from google.colab import files
        files.download(path)
        print("Downloaded:", path)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)

In [ ]:
import os, platform, subprocess, time

def sh(cmd):
    p = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        encoding="utf-8", errors="replace")
    return p.stdout.strip()

print("hostname:", platform.node() or "n/a")
print("os:", platform.platform())
print("nproc:", sh("nproc"))
print("cpu:", sh("grep -m1 -E 'model name' /proc/cpuinfo") or "n/a")
print(sh("free -g") or "RAM information unavailable")
print("mc.cores is fixed at 2; model fits are fixed to one thread.")


In [ ]:
import subprocess
p = subprocess.run(
    ["bash", "-lc", "apt-get -qq update >/dev/null && "
     "apt-get -qq install -y --no-install-recommends "
     "r-base r-base-dev libcurl4-openssl-dev >/dev/null 2>&1"],
    capture_output=True, text=True, encoding="utf-8", errors="replace")
if p.returncode != 0:
    print(p.stdout[-1000:])
    print(p.stderr[-2000:])
    raise RuntimeError("R installation failed")
print(subprocess.check_output(
    ["R", "--version"], text=True,
    encoding="utf-8", errors="replace").splitlines()[0])


In [ ]:
import hashlib, os, re, shutil, subprocess, sys
from pathlib import Path

BUNDLE_FOLDER_URL = 'https://drive.google.com/drive/folders/1w3quuskj25CBOFCGG0mTRGUHcufPpdb3?usp=sharing'
BUNDLE_SHA256 = '12d223bc0fcef624c1ff4cc35c5d7ecc1b1f9b05aa84ecd9d9e4a5a3382bae3c'
assert re.fullmatch(r"[0-9a-fA-F]{64}", BUNDLE_SHA256), \
    "Bundle SHA256 is missing or malformed; regenerate the notebook."

BUNDLE_DOWNLOAD_DIR = Path("/content/tisca_bundle_download")
if BUNDLE_DOWNLOAD_DIR.exists():
    shutil.rmtree(BUNDLE_DOWNLOAD_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True,
)
# NOT `download`: that name is the Colab-download helper defined in the
# setup cell, and binding a CompletedProcess to it here made the final
# cell of E3_seed_verification.ipynb die with "'CompletedProcess' object
# is not callable" after a 50-minute run had already succeeded.
gdown_proc = subprocess.run(
    [sys.executable, "-m", "gdown", "--folder", BUNDLE_FOLDER_URL,
     "--output", str(BUNDLE_DOWNLOAD_DIR), "--remaining-ok"],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace",
)
print(gdown_proc.stdout[-4000:])
if gdown_proc.returncode != 0:
    print(gdown_proc.stderr[-4000:])
    raise RuntimeError("Google Drive bundle download failed")
tar_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.tar.gz"))
sha_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.sha256"))
assert len(tar_candidates) == 1, f"expected one tarball, found {tar_candidates}"
assert len(sha_candidates) <= 1, f"expected at most one checksum file, found {sha_candidates}"
if sha_candidates:
    published_sha = sha_candidates[0].read_text().split()[0].lower()
    assert published_sha == BUNDLE_SHA256.lower(), \
        "published tisca_rlib.sha256 differs from the generated notebook"
    print("verified published checksum sidecar:", sha_candidates[0])
else:
    print("no tisca_rlib.sha256 sidecar in the shared folder; "
          "verifying the tarball against the embedded SHA256")
download_path = "/content/_dl_tisca_rlib.tar.gz"
shutil.copy2(tar_candidates[0], download_path)
with open(download_path, "rb") as f:
    observed_sha = hashlib.sha256(f.read()).hexdigest()
assert observed_sha == BUNDLE_SHA256.lower(), "R library bundle SHA mismatch"
if os.path.isdir("/content/tisca_rlib"):
    shutil.rmtree("/content/tisca_rlib")
subprocess.run(["tar", "xzf", download_path, "-C", "/content"], check=True)
LIBDIR = "/content/tisca_rlib/rlib"
assert os.path.isdir(LIBDIR), "bundle did not restore the expected rlib directory"
print("bundle restored:", LIBDIR, "from", BUNDLE_FOLDER_URL)


In [ ]:
import os, urllib.request

RUNCELL_URL = (
    "https://raw.githubusercontent.com/hugogobato/"
    "Test-Informed-Simulation-Count-Algorithm-TISCA/main/"
    "experiments/E3_mvbcf_casestudy/run_cell.R"
)
MVBCF_CPP_URL = (
    "https://raw.githubusercontent.com/Nathan-McJames/MVBCF_Paper/"
    "main/MVBCF_Code.cpp"
)
os.makedirs("/content/e3", exist_ok=True)
urllib.request.urlretrieve(RUNCELL_URL, "/content/e3/run_cell.R")
# The upstream C++ is downloaded at runtime and is never committed here.
urllib.request.urlretrieve(MVBCF_CPP_URL, "/content/e3/MVBCF_Code.cpp")
assert os.path.getsize("/content/e3/run_cell.R") > 1000
assert os.path.getsize("/content/e3/MVBCF_Code.cpp") > 10000
with open("/content/e3/run_cell.R") as f:
    run_cell_source = f.read()
required_fixes = [
    "nthread = nthread_global",
    "num_threads = nthread_global",
    "acquired <- dir.create(lk",
    "num_gfr = 0",
    "sigma2_leaf_init = 1^2 / n_tree_mu",
    "sigma2_leaf_init = 0.375^2 / n_tree_tau",
    'propensity_covariate = "prognostic"',
    "sample_sigma2_leaf = FALSE",
]
missing_fixes = [item for item in required_fixes if item not in run_cell_source]
assert not missing_fixes, (
    "GitHub main is serving a stale run_cell.R. Commit and push the "
    f"corrected driver before running this notebook; missing: {missing_fixes}"
)
print("downloaded run_cell.R and upstream MVBCF_Code.cpp")


In [ ]:
import os, subprocess

compile_script = "\n".join([
    ".libPaths(c('/content/tisca_rlib/rlib', .libPaths()))",
    "if (!requireNamespace('Rcpp', quietly=TRUE) ||",
    "    !requireNamespace('RcppArmadillo', quietly=TRUE) ||",
    "    !requireNamespace('RcppDist', quietly=TRUE)) stop('bundle missing Rcpp dependencies')",
    "library(Rcpp)",
    "sourceCpp('/content/e3/MVBCF_Code.cpp')",
    "stopifnot(is.function(fast_bart))",
    "cat('FAST_BART_OK\\n')",
])
with open("/content/e3/compile.R", "w") as f:
    f.write(compile_script)
p = subprocess.run(["Rscript", "/content/e3/compile.R"],
                   capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:])
if p.returncode != 0 or "FAST_BART_OK" not in p.stdout:
    print(p.stderr[-3000:])
    raise RuntimeError("upstream MVBCF C++ compilation failed")
print("fast_bart() compiled")


## The probe driver

Standalone on purpose: it does not import, source or modify `run_cell.R`, so
nothing here can perturb the driver that produced the confirmatory shards.
The DGP block and every model call are copied verbatim from it, so the answer
is about this study's fits rather than about the libraries in the abstract.

In [ ]:
PROBE_R_SOURCE = r"""
suppressWarnings(suppressMessages({
  library(dbarts); library(bartCause); library(stochtree); library(mvtnorm)
}))

RNGkind("L'Ecuyer-CMRG")

# ---- settings, verbatim from run_cell.R:217-221 ---------------------------- #
n_train <- 100L
n_test  <- 1000L
n_tree_mu <- 50L; n_tree_tau <- 20L; n_trees_total <- n_tree_mu + n_tree_tau
n_iter <- 1000L; n_burn <- 500L; n_mcmc <- n_iter - n_burn
nthread_global <- 1L

# ---- DGP1, verbatim from run_cell.R:139-199 -------------------------------- #
generate_data <- function(n) {
  X1 <- runif(n); X2 <- runif(n); X3 <- runif(n); X4 <- runif(n); X5 <- runif(n)
  X6 <- rbinom(n, 1, 0.5); X7 <- rbinom(n, 1, 0.5); X8 <- rbinom(n, 1, 0.5)
  X9 <- sample(c(0, 1, 2, 3, 4), n, replace = TRUE)
  X10 <- sample(c(0, 1, 2, 3, 4), n, replace = TRUE)
  X <- cbind(X1, X2, X3, X4, X5, X6, X7, X8, X9, X10)
  Mu1 <- (11*sin(pi*X1*X2)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
  Mu2 <- (9*sin(pi*X1*X2)+22*(X3-0.5)^2+14*X4+8*X6+X9)*10+300
  Tau1 <- (2*X4+2*X5)*10
  Tau2 <- (1*X4+3*X5)*10
  Z <- rbinom(n, 1, X4)
  Y <- cbind(Mu1 + Z*Tau1, Mu2 + Z*Tau2) +
    mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow = 2, byrow = TRUE))
  list(X = X, Y = Y, Z = Z)
}

paper_lambda <- function(y, z, x_control, pihat, nu = 3, sigq = 0.9) {
  ystd <- (y - mean(y)) / sd(y)
  x_c <- cbind(x_control, pihat)
  sighat <- summary(lm(ystd ~ z + as.matrix(x_c)))$sigma
  (sighat^2 * qchisq(1 - sigq, nu)) / nu
}

set_rand <- function(x) assign(".Random.seed", x, envir = .GlobalEnv)

# Two global RNG states that are certainly different: independent L'Ecuyer
# substreams, which is what run_cell.R hands to consecutive replications. If a fit
# is invariant across these two, it is invariant across the whole design.
set.seed(11L); STATE_A <- .Random.seed
STATE_B <- parallel::nextRNGStream(parallel::nextRNGStream(STATE_A))
stopifnot(!identical(STATE_A, STATE_B))

SEED_1 <- 424242L   # the explicit seed held fixed across STATE_A / STATE_B
SEED_2 <- 989898L   # a different explicit seed, for the positive control

# ---- data: generated ONCE, from a state neither probe state uses ----------- #
set.seed(20260808L)
d <- generate_data(n_train)
X <- d$X; Y <- d$Y; Z <- d$Z
td <- generate_data(n_test)
X_test <- td$X; Z_test <- td$Z

# ---- fits, each a closure of (explicit seed, global state) ------------------ #
# Each returns a numeric vector: the object run_cell.R actually turns into the
# recorded metrics, flattened. Comparing the full posterior object rather than a
# scalar summary means a difference cannot hide behind an averaging step.

fit_dbarts <- function(seed_val, state) {
  set_rand(state)
  m <- dbarts::bart(x.train = X, y.train = Z, x.test = X_test, k = 3,
                    nthread = nthread_global, seed = seed_val, verbose = FALSE)
  as.numeric(m$yhat.test)
}

fit_bcf <- function(seed_val, state) {
  set_rand(state)
  # The propensity is fixed input here, not part of what is under test: compute it
  # once, deterministically, so a difference cannot be inherited from upstream.
  p <- P_FIXED; p_test <- P_TEST_FIXED
  lambda <- paper_lambda(Y[, 1], Z, X, p)
  m <- stochtree::bcf(
    X_train = X, Z_train = Z, y_train = Y[, 1],
    propensity_train = p, X_test = X_test, Z_test = Z_test,
    propensity_test = p_test, num_gfr = 0,
    num_burnin = n_burn, num_mcmc = n_mcmc,
    general_params = list(
      standardize = TRUE, sample_sigma2_global = TRUE,
      sigma2_global_shape = 3 / 2,
      sigma2_global_scale = if (is.finite(lambda)) 3 * lambda / 2 else 1,
      propensity_covariate = "prognostic", adaptive_coding = FALSE,
      num_chains = 1, num_threads = nthread_global, random_seed = seed_val),
    prognostic_forest_params = list(
      num_trees = n_tree_mu, alpha = 0.95, beta = 2,
      min_samples_leaf = 1, max_depth = -1,
      sample_sigma2_leaf = FALSE,
      sigma2_leaf_init = 1^2 / n_tree_mu),
    treatment_effect_forest_params = list(
      num_trees = n_tree_tau, alpha = 0.25, beta = 3,
      min_samples_leaf = 1, max_depth = -1,
      sample_sigma2_leaf = FALSE,
      sigma2_leaf_init = 0.375^2 / n_tree_tau))
  as.numeric(m$tau_hat_test)
}

fit_bartc <- function(seed_val, state) {
  set_rand(state)
  Xa <- cbind(X, P_FIXED)
  colnames(Xa) <- paste0("V", seq_len(ncol(Xa)))
  Xt <- cbind(X_test, P_TEST_FIXED)
  colnames(Xt) <- colnames(Xa)
  m <- bartCause::bartc(Y[, 1], Z, Xa, p.scoreAsCovariate = FALSE, n.chains = 1,
                        n.threads = 1, keepTrees = TRUE, n.trees = n_trees_total,
                        seed = seed_val, verbose = FALSE)
  as.numeric(predict(m, Xt, type = "icate"))
}

# Propensity, computed once from a third state so every later fit sees the same
# numbers. This is input to bcf and bartc, so it must not vary between arms.
set_rand(STATE_A)
p_mod <- dbarts::bart(x.train = X, y.train = Z, x.test = X_test, k = 3,
                      nthread = nthread_global, seed = 7777L, verbose = FALSE)
P_FIXED <- colMeans(pnorm(p_mod$yhat.train))
P_TEST_FIXED <- colMeans(pnorm(p_mod$yhat.test))

MODELS <- list(
  "dbarts::bart (propensity fit)" = fit_dbarts,
  "stochtree::bcf"                = fit_bcf,
  "bartCause::bartc (dbarts)"     = fit_bartc)

rows <- list()
for (nm in names(MODELS)) {
  f <- MODELS[[nm]]
  cat("fitting:", nm, "\n"); flush.console()
  t0 <- proc.time()[["elapsed"]]
  a1 <- f(SEED_1, STATE_A)          # reference
  a2 <- f(SEED_1, STATE_B)          # same explicit seed, DIFFERENT global state
  ctrl <- f(SEED_2, STATE_A)        # different explicit seed, same global state
  stopifnot(length(a1) == length(a2), length(a1) == length(ctrl))
  d_state <- max(abs(a1 - a2))
  d_seed  <- max(abs(a1 - ctrl))
  rows[[length(rows) + 1L]] <- data.frame(
    model = nm,
    n_values = length(a1),
    max_abs_diff_same_seed_other_state = d_state,
    max_abs_diff_other_seed_same_state = d_seed,
    invariant_to_global_state = identical(a1, a2),
    sensitive_to_explicit_seed = !identical(a1, ctrl),
    fit_seconds = round(proc.time()[["elapsed"]] - t0, 1),
    stringsAsFactors = FALSE)
}
out <- do.call(rbind, rows)
out$verdict <- ifelse(!out$sensitive_to_explicit_seed, "INCONCLUSIVE",
               ifelse(out$invariant_to_global_state, "HONOURS_SEED",
                      "ALSO_USES_GLOBAL_STREAM"))
write.csv(out, OUT_PATH, row.names = FALSE)
print(out)
cat("PROBE_OK\n")
"""

In [ ]:
import os
import subprocess
import time

import pandas as pd

os.makedirs("/content/e3", exist_ok=True)
OUT_CSV = "/content/e3/seed_honouring.csv"
script = ('OUT_PATH <- "%s"\n' % OUT_CSV) + PROBE_R_SOURCE
with open("/content/e3/seed_probe.R", "w") as f:
    f.write(script)

env = dict(os.environ)
env["R_LIBS"] = LIBDIR + ":" + env.get("R_LIBS", "")

t0 = time.time()
p = subprocess.run(["Rscript", "/content/e3/seed_probe.R"],
                   capture_output=True, text=True,
                   encoding="utf-8", errors="replace", env=env)
print(p.stdout[-6000:])
print(f"probe wall-clock: {time.time() - t0:.0f}s")
if p.returncode != 0 or "PROBE_OK" not in p.stdout:
    print(p.stderr[-6000:])
    raise RuntimeError("seed probe failed; do not tick the 3.6 box")

probe = pd.read_csv(OUT_CSV)
print()
print(probe.to_string(index=False))

## Verdict, and the block to paste into CALIBRATION.md

In [ ]:
# --------------------------------------------------------------------------- #
# Verdict                                                                       #
# --------------------------------------------------------------------------- #
# Three outcomes, and only one of them is a defect-shaped finding:
#
#   HONOURS_SEED             the fit is bit-identical at a fixed explicit seed across
#                            two different global RNG states, and differs when the
#                            seed changes. The seed argument alone determines the fit.
#   ALSO_USES_GLOBAL_STREAM  the fit responds to its seed BUT also consumes the global
#                            stream. Normal for an R model wrapper, and precisely the
#                            case docs/seed_rng_protocol.md 3.3 positioning covers.
#                            run_cell.R sets that stream deterministically per fit, so
#                            reproducibility is unaffected -- the ATTRIBUTION changes,
#                            not the guarantee.
#   INCONCLUSIVE             two different explicit seeds gave identical output, so
#                            the fit is not seed-sensitive at all and this comparison
#                            has no power. Investigate before reporting anything.
#
# fast_bart() (MVBCF) and MultiskewBART are absent by construction: neither exposes a
# seed argument, so there is nothing for them to honour. They are governed entirely by
# global stream positioning, which the 3.5 four-way identity test already validated.

probe["attribution"] = probe["verdict"].map({
    "HONOURS_SEED": "explicit seed argument",
    "ALSO_USES_GLOBAL_STREAM": "global stream positioning (run_cell.R 3.3)",
    "INCONCLUSIVE": "UNDETERMINED -- comparison had no power",
})
inconclusive = probe[probe["verdict"] == "INCONCLUSIVE"]

os.makedirs(os.path.join(RESULTS, "E3"), exist_ok=True)
dest = os.path.join(RESULTS, "E3", "seed_honouring.csv")
probe.to_csv(dest, index=False)
download(dest)

print(probe[["model", "verdict", "attribution",
             "max_abs_diff_same_seed_other_state",
             "max_abs_diff_other_seed_same_state"]].to_string(index=False))
print()
if len(inconclusive):
    print("*** INCONCLUSIVE rows above: do NOT tick the 3.6 box for them ***")
else:
    print("Every model under test responded to its explicit seed, so the "
          "comparison had power and the verdicts stand.")

honours = probe.loc[probe["verdict"] == "HONOURS_SEED", "model"].tolist()
global_too = probe.loc[probe["verdict"] == "ALSO_USES_GLOBAL_STREAM", "model"].tolist()

print()
print("--- paste into experiments/E3_mvbcf_casestudy/CALIBRATION.md ---")
print(f"""
### P0-T2 3.6, per-model seed honouring, {pd.Timestamp.today().date()}

Run by `notebooks/E3_seed_honouring.ipynb`; table in
`results/E3/seed_honouring.csv`. One DGP1 n = 100 dataset, nine fits, no stored
replication read or re-run.

- [{'x' if not len(inconclusive) else ' '}] Discriminating comparison performed:
      same model, same data, same explicit seed, two different global RNG states,
      with a different-seed positive control confirming the comparison had power.
- Determined by the explicit seed argument alone: {honours or 'none'}.
- Responds to the seed but also consumes the global stream: {global_too or 'none'}.
      Not a defect: `run_cell.R` positions a fixed L'Ecuyer substream before every
      fit, which is what 3.3 requires and what the 3.5 four-way identity test
      already validated on the real driver.
- Out of scope by construction: `fast_bart()` (MVBCF) and `MultiskewBART` expose no
      seed argument, so `model_seed_mvbcf` and `model_seed_mvbart` are recorded
      labels rather than arguments passed. Their reproducibility rests entirely on
      3.3 positioning.

Data Availability wording this licenses: reproducibility of the shipped rows is
established empirically (three seeds, ten hosts, 159 columns, zero differences) and
is attributed to deterministic global-stream positioning{', with the seed argument sufficient on its own for ' + ', '.join(honours) if honours else ''}.
""")